# 문항 1 컨텐츠 기반 필터링

In [45]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

import warnings; warnings.filterwarnings('ignore')

movies = pd.read_csv('./movielens/tmdb_5000_movies.csv')
movies.shape

(4803, 20)

In [46]:
movies.head(2).T

,0,1
budget,237000000,300000000
genres,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
homepage,http://www.avatarmovie.com/,http://disney.go.com/disneypictures/pirates/
id,19995,285
keywords,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
original_language,en,en
original_title,Avatar,Pirates of the Caribbean: At World's End
overview,"In the 22nd century, a paraplegic Marine is di...","Captain Barbossa, long believed to be dead, ha..."
popularity,150.437577,139.082615
production_companies,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""..."


In [47]:
indices = pd.Series(movies.index, index=movies['title'])

In [48]:
movies['genres'] = movies['genres'].apply(eval)
movies['keywords'] = movies['keywords'].apply(eval)

In [49]:
movies['genres_literal'] = movies['genres'].apply(lambda x: [i['name'] for i in x]).apply(lambda x: ' '.join(x))

In [50]:
vect = CountVectorizer(min_df=1, ngram_range=(1, 2))
genre_mat = vect.fit_transform(movies['genres_literal'])

In [51]:
genre_mat.shape

(4803, 276)

In [52]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim = cosine_similarity(genre_mat, genre_mat)
genre_sim_indices = genre_sim.argsort()[:, ::-1]

In [53]:
def find_similar_movies(df, ind, title, top_n=10):
    idx = indices[title]
    sim_idx = ind[idx, :(top_n)]
    sim_idx = sim_idx.reshape(-1)
    return df.iloc[sim_idx]

top10_movies = find_similar_movies(movies, genre_sim_indices,'Avatar', top_n=10)
top10_movies['title']

0                                          Avatar
46                     X-Men: Days of Future Past
3494    Beastmaster 2: Through the Portal of Time
870                                   Superman II
14                                   Man of Steel
813                                      Superman
1652                         Dragonball Evolution
1296                                 Superman III
420                   Hellboy II: The Golden Army
419                                        Jumper
Name: title, dtype: str

# 문항 2 아이템 기반 최근접 이웃 협업 필터링

In [68]:
movies = pd.read_csv('./movielens/movies.csv')
ratings = pd.read_csv('./movielens/ratings.csv')
movies.shape, ratings.shape

((9742, 3), (100836, 4))

In [65]:
movies.head(2)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy


In [66]:
ratings.head(2)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247


In [67]:
merged_df = pd.merge(movies, ratings, on='movieId')
merged_df.head(2)

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962


In [69]:
rating_matrix = merged_df.pivot_table(index='userId', columns='title', values='rating').fillna(0)
rating_matrix.head(2)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [70]:
rating_matrix_T = rating_matrix.T

In [72]:
rating_sim = cosine_similarity(rating_matrix_T, rating_matrix_T)
rating_sim_df = pd.DataFrame(rating_sim, index=rating_matrix.columns, columns=rating_matrix.columns)
rating_sim_df.head(2)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.0,0.707107,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


In [75]:
def get_similar_movies(df, title, top_n=10):
    similar_movies = df[title].sort_values(ascending=False).drop(title)
    return similar_movies.iloc[:top_n]

In [77]:
get_similar_movies(rating_sim_df, 'Godfather, The (1972)')

title
Godfather: Part II, The (1974)                           0.821773
Goodfellas (1990)                                        0.664841
One Flew Over the Cuckoo's Nest (1975)                   0.620536
Star Wars: Episode IV - A New Hope (1977)                0.595317
Fargo (1996)                                             0.588614
Star Wars: Episode V - The Empire Strikes Back (1980)    0.586030
Fight Club (1999)                                        0.581279
Reservoir Dogs (1992)                                    0.579059
Pulp Fiction (1994)                                      0.575270
American Beauty (1999)                                   0.575012
Name: Godfather, The (1972), dtype: float64